# 01 — Exploratory Data Analysis\n\nDataset: `data/preprocessed/newsletters_preprocessed.csv` (1,654 rows, full unsplit dataset)\n\nGoal: understand the dataset before building classifiers — class balance, text lengths, source coverage, gaps.

In [ ]:
import pandas as pd\nimport matplotlib.pyplot as plt\nimport seaborn as sns\n\nsns.set_theme(style="whitegrid")\npd.set_option("display.max_colwidth", 80)\n\ndf = pd.read_csv("../data/preprocessed/newsletters_preprocessed.csv")\nprint(f"Shape: {df.shape}")\ndf.head(3)

In [ ]:
df.info()

## 1. Class distribution (`new_theme`)\n\nHow imbalanced are the 6 sections? This directly affects classifier strategy (e.g. class weights, oversampling).

In [ ]:
theme_counts = df["new_theme"].value_counts()\ntheme_pct = df["new_theme"].value_counts(normalize=True).mul(100).round(1)\n\nfig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\ntheme_counts.plot.barh(ax=axes[0], color="steelblue")\naxes[0].set_title("Items per section")\naxes[0].set_xlabel("Count")\n\ntheme_pct.plot.barh(ax=axes[1], color="coral")\naxes[1].set_title("Section share (%)")\naxes[1].set_xlabel("%")\n\nplt.tight_layout()\nplt.show()\n\npd.DataFrame({"count": theme_counts, "pct": theme_pct})

## 2. Item type breakdown\n\nWhat proportion are news articles vs reports vs academic papers? This tells us what kind of text the classifier will see.

In [ ]:
item_counts = df["item_type"].value_counts()\n\nfig, ax = plt.subplots(figsize=(10, 5))\nitem_counts.plot.barh(ax=ax, color="teal")\nax.set_title("Item type distribution")\nax.set_xlabel("Count")\nplt.tight_layout()\nplt.show()\n\nitem_counts

## 3. Source sectors (`org_broad_category`)\n\nWhich sectors dominate the newsletter content?

In [ ]:
sector_counts = df["org_broad_category"].value_counts()\n\nfig, ax = plt.subplots(figsize=(10, 5))\nsector_counts.plot.barh(ax=ax, color="mediumpurple")\nax.set_title("Items by source sector")\nax.set_xlabel("Count")\nplt.tight_layout()\nplt.show()\n\nsector_counts

## 4. Top organisations by article count

In [ ]:
top_orgs = df["organisation"].value_counts().head(20)\n\nfig, ax = plt.subplots(figsize=(10, 6))\ntop_orgs.plot.barh(ax=ax, color="darkorange")\nax.set_title("Top 20 organisations by article count")\nax.set_xlabel("Count")\nax.invert_yaxis()\nplt.tight_layout()\nplt.show()\n\ntop_orgs

## 5. Title and description length distributions\n\nAre titles long enough to classify from? This is critical since the classifier will use title (+ snippet) at inference time.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))\n\ndf["title_length"].hist(bins=40, ax=axes[0], color="steelblue", edgecolor="white")\naxes[0].set_title("Title length (chars)")\naxes[0].set_xlabel("Characters")\naxes[0].axvline(df["title_length"].median(), color="red", linestyle="--", label=f'median={df["title_length"].median():.0f}')\naxes[0].legend()\n\ndf["text_length_words"].hist(bins=40, ax=axes[1], color="coral", edgecolor="white")\naxes[1].set_title("Text length (words) — title + description")\naxes[1].set_xlabel("Words")\naxes[1].axvline(df["text_length_words"].median(), color="red", linestyle="--", label=f'median={df["text_length_words"].median():.0f}')\naxes[1].legend()\n\nplt.tight_layout()\nplt.show()\n\ndf[["title_length", "description_length", "text_length_words"]].describe().round(1)

## 6. Articles per newsletter issue over time\n\nAny gaps or spikes? Helps understand data consistency.

In [ ]:
items_per_issue = df.groupby("newsletter_number").size()\n\nfig, ax = plt.subplots(figsize=(14, 5))\nitems_per_issue.plot(ax=ax, marker="o", markersize=3, linewidth=1, color="steelblue")\nax.set_title("Items per newsletter issue")\nax.set_xlabel("Newsletter number")\nax.set_ylabel("Number of items")\nax.axhline(items_per_issue.median(), color="red", linestyle="--", alpha=0.6, label=f"median={items_per_issue.median():.0f}")\nax.legend()\nplt.tight_layout()\nplt.show()\n\nprint(f"Issues covered: {items_per_issue.index.min()} to {items_per_issue.index.max()}")\nprint(f"Median items/issue: {items_per_issue.median():.0f}")\nprint(f"Min: {items_per_issue.min()}, Max: {items_per_issue.max()}")

## 7. Unmapped organisations / domains\n\nHow many items have no mapped organisation? Are there important sources missing from the lookup?

In [ ]:
unmapped = df["organisation"].isna().sum()\ntotal = len(df)\nprint(f"Unmapped organisations: {unmapped} / {total} ({unmapped/total*100:.1f}%)")\n\n# Top unmapped domains — these might need adding to the lookup\nunmapped_domains = df.loc[df["organisation"].isna(), "domain"].value_counts().head(15)\nprint(f"\\nTop 15 unmapped domains:")\nunmapped_domains

## 8. Section × item type cross-tabulation\n\nDo certain sections favour particular item types? (e.g. `political_environment` heavy on news, `research_practice_policy` heavy on academic articles)

In [ ]:
ct = pd.crosstab(df["new_theme"], df["item_type"])\n\nfig, ax = plt.subplots(figsize=(12, 6))\nct.plot.barh(stacked=True, ax=ax, colormap="tab10")\nax.set_title("Item type by section")\nax.set_xlabel("Count")\nax.legend(bbox_to_anchor=(1.05, 1), loc="upper left")\nplt.tight_layout()\nplt.show()\n\nct

## 9. Train / val split check\n\nConfirm the split preserved class proportions.

In [ ]:
train = pd.read_csv("../data/processed/train.csv")\nval = pd.read_csv("../data/processed/val.csv")\n\nprint(f"Train: {len(train)} rows, Val: {len(val)} rows")\nprint(f"Split ratio: {len(train)/(len(train)+len(val))*100:.1f}% / {len(val)/(len(train)+len(val))*100:.1f}%\\n")\n\ncompare = pd.DataFrame({\n    "train_pct": train["new_theme"].value_counts(normalize=True).mul(100).round(1),\n    "val_pct": val["new_theme"].value_counts(normalize=True).mul(100).round(1),\n})\ncompare["diff"] = (compare["train_pct"] - compare["val_pct"]).abs()\ncompare

## 10. Key takeaways\n\n*Run the notebook and fill in observations here:*\n\n- Class balance: ...\n- Title lengths: ...\n- Dominant sources/sectors: ...\n- Unmapped coverage: ...\n- Anything surprising: ...